# Pothole Detection | Hybrid Models
**Models:** YOLOv8m+CBAM, YOLOv8m+CoordAttn, YOLO+FasterRCNN Ensemble 

In [1]:
# Frees GPU memory by deleting any model objects left in the namespace from a previous run and emptying the CUDA cache.
import torch, gc

# Delete any existing models in memory
for var in [
    "cbam_yolo",
    "ca_yolo",
    "yolo_ens",
    "frcnn_ens",
    "cbam_injector",
    "ca_injector",
    "test_model",
    "test_model_cbam",
    "test_model_ca",
]:
    if var in dir():
        del var

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

# Check memory after clearing
print("GPU Memory after clearing:")
print(f"  Allocated : {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"  Reserved  : {torch.cuda.memory_reserved() / 1024**2:.1f} MB")
print(
    f"  Free      : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**2:.1f} MB"
)
print(
    f"  Total     : {torch.cuda.get_device_properties(0).total_memory / 1024**2:.1f} MB"
)

GPU Memory after clearing:
  Allocated : 0.0 MB
  Reserved  : 0.0 MB
  Free      : 24251.6 MB
  Total     : 24251.6 MB


In [2]:
2  # Prints the GPU name to confirm which device the notebook is running on.
import torch

print(f"   GPU: {torch.cuda.get_device_name(0)}")

   GPU: NVIDIA GeForce RTX 3090


In [4]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR, HYBRID_DIR and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/saved_models"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
    SAVE_DIR = "/content/saved_models"
else:
    print("Running on LOCAL JUPYTER")
    ROOT = "."
    SAVE_DIR = "./saved_models"

OUTPUT_DIR = os.path.join(ROOT, "hybrid_results")
DATA_DIR = os.path.join(ROOT, "data")
HYBRID_DIR = os.path.join(ROOT, "hybrid_models")

for d in [OUTPUT_DIR, DATA_DIR, HYBRID_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"   Output       -> {OUTPUT_DIR}")
print(f"   Base Weights -> {SAVE_DIR}")
print(f"   Hybrid Saves -> {HYBRID_DIR}")

Running on LOCAL JUPYTER
   Output       -> ./hybrid_results
   Base Weights -> ./saved_models
   Hybrid Saves -> ./hybrid_models


In [5]:
# Loads all shared imports and defines the fixed evaluation config, DEVICE, CONF_THRESH, IOU_THRESH, IMG_SIZE, and the chart palette.
import time, json, glob, warnings, xml.etree.ElementTree as ET
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMG_SIZE = 640
MAX_IMAGES = 150
NUM_WORKERS = (
    2  # cluster-safe Ultralytics/torchvision defaults (8) can exceed cluster ulimits
)


EPOCHS_FROZEN = 10  # phase 1: backbone frozen, attention learns fast
LR_FROZEN = 1e-3
EPOCHS_FULL = 40  # phase 2: everything trains together, gently
LR_FULL = 2e-4
WEIGHT_DECAY = 5e-4

# Confidence grid for the F1 operating-point sweep
CONF_SWEEP = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]

# Paths to our fine-tuned base weights
YOLOV8M_WEIGHTS = f"{SAVE_DIR}/YOLOv8m_finetuned.pt"
FASTERRCNN_WEIGHTS = f"{SAVE_DIR}/faster_rcnn_finetuned.pth"

PALETTE = {
    "YOLOv8m+CBAM": "#f59e0b",
    "YOLOv8m+CoordAttn": "#10b981",
    "YOLO+FRCNN Ensemble": "#8b5cf6",
    "YOLOv8m (base)": "#00d4ff",
    "Faster R-CNN (base)": "#f97316",
}

print(f" Config loaded  |  Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(
    f"   Two-phase: {EPOCHS_FROZEN}ep @ lr={LR_FROZEN} (freeze=10) "
    f"-> {EPOCHS_FULL}ep @ lr={LR_FULL} (freeze=0)"
)

# Quick weight file check
for name, path in [("YOLOv8m", YOLOV8M_WEIGHTS), ("Faster R-CNN", FASTERRCNN_WEIGHTS)]:
    exists = os.path.exists(path)
    print(f"   {name} weights: {'FOUND' if exists else 'NOT FOUND'} at {path}")


 Config loaded  |  Device: cuda
   GPU: NVIDIA GeForce RTX 3090
   Two-phase: 10ep @ lr=0.001 (freeze=10) -> 40ep @ lr=0.0002 (freeze=0)
   YOLOv8m weights: FOUND at ./saved_models/YOLOv8m_finetuned.pt
   Faster R-CNN weights: FOUND at ./saved_models/faster_rcnn_finetuned.pth


In [6]:
# Downloads the three Kaggle datasets, loads them through the unified annotation loader, deduplicates, and defines the canonical train/val split shared with the other notebooks.
import kagglehub
from pathlib import Path

print(" Downloading datasets via kagglehub ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")

DATASET_ROOTS = {"chitholian": path_1, "andrewmvd": path_2, "ashishkumar": path_3}
print("Datasets ready")


import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from PIL import Image


def load_annotated_potholes(root):
    root = Path(root)
    records = []
    for img_path in list(root.rglob("*.jpg")) + list(root.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if not xml_path.exists():
            xml_path = img_path.parent.parent / "annotations" / (img_path.stem + ".xml")
        gt_boxes = []
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        {
                            "label": (obj.find("name").text or "pothole").lower(),
                            "xmin": float(bb.find("xmin").text),
                            "ymin": float(bb.find("ymin").text),
                            "xmax": float(bb.find("xmax").text),
                            "ymax": float(bb.find("ymax").text),
                        }
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


def load_ashishkumar_csv(root):
    root = Path(root)
    csv_path = root / "train" / "labels.csv"
    img_dir = root / "train" / "images"
    df = pd.read_csv(csv_path)
    grouped = df.groupby("ImageID")

    records = []
    for img_path in sorted(img_dir.glob("*.jpg")):
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    {
                        "label": "pothole",
                        "xmin": float(row["XMin"]),
                        "ymin": float(row["YMin"]),
                        "xmax": float(row["XMax"]),
                        "ymax": float(row["YMax"]),
                    }
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


all_records = []
for name, root in DATASET_ROOTS.items():
    if name == "ashishkumar":
        recs = load_ashishkumar_csv(root)
    else:
        recs = load_annotated_potholes(root)
    print(
        f"  {name}: {len(recs)} images ({sum(len(r['gt_boxes']) for r in recs)} gt boxes)"
    )
    all_records.extend(recs)

print(f"\nTotal images before dedup: {len(all_records)}")

# Normalized-pixel nearest-neighbor dedup (robust to re-encoding differences)
NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0  # mean abs pixel diff (0-255 scale), true duplicates
# measured at 0.10-0.50, unrelated images much higher


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


annotated_only = [r for r in all_records if r["gt_boxes"]]
print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in annotated_only])

keep_mask = np.ones(len(annotated_only), dtype=bool)
seen_arrs = []
for i in range(len(annotated_only)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

records = [r for r, keep in zip(annotated_only, keep_mask) if keep]

n_before = len(all_records)
n_after = len(records)
print(f"Total images after dedup: {n_after}")
print(f"   Duplicates removed: {n_before - n_after}")


import random as _random

_random.seed(42)
_shuffled = records.copy()
_random.shuffle(_shuffled)
_split_idx = int(len(_shuffled) * 0.8)
train_recs = _shuffled[:_split_idx]
val_recs = _shuffled[_split_idx:]
print(
    f"\nCanonical split: train={len(train_recs)}  val={len(val_recs)}  "
    f"(seed=42, shared by all training and evaluation cells below)"
)


Datasets ready
  chitholian: 665 images (1740 gt boxes)
  andrewmvd: 665 images (1740 gt boxes)
  ashishkumar: 674 images (1371 gt boxes)

Total images before dedup: 2004
Computing normalized pixel arrays for dedup ...
Total images after dedup: 926
   Duplicates removed: 1078

Canonical split: train=740  val=186  (seed=42, shared by all training and evaluation cells below)
